# SWaT / WADI：冻结 C3 restricted-state fusion FiLM

每个数据集四臂：Baseline、无辅助 FiLM、真实配对 FiLM、打乱配对 FiLM。关闭 Transformer、关系联合评分与旧 relation-transition 分支。

In [ ]:
from pathlib import Path
import hashlib, os, shutil, subprocess, sys, tarfile, torch
assert torch.cuda.is_available(), '请为 notebook 启用 GPU'
work_root = Path('/kaggle/working/EnhancedMTADGAT')
archives = list(Path('/kaggle/input').rglob('enhancedmtadgat-frozen-source.tar.gz'))
if len(archives) == 1:
    expected_sha = archives[0].with_name('source-sha256.txt').read_text().split()[0]
    actual_sha = hashlib.sha256(archives[0].read_bytes()).hexdigest()
    assert actual_sha == expected_sha, (actual_sha, expected_sha)
    work_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archives[0], 'r:gz') as archive:
        members = archive.getmembers()
        assert all(not Path(m.name).is_absolute() and '..' not in Path(m.name).parts for m in members)
        archive.extractall(work_root, members=members)
    source_label = str(archives[0])
else:
    plans = list(Path('/kaggle/input').rglob('89_swat_wadi_frozen_c3_restricted_film.json'))
    source_roots = [p.parents[2] for p in plans if (p.parents[2] / 'run.py').is_file()]
    assert len(source_roots) == 1, {'archives': archives, 'plans': plans}
    shutil.copytree(source_roots[0], work_root, dirs_exist_ok=True)
    actual_sha = '953d7a43ee03ccf6e258c91d4bb846f00ed4691f5f9ce83298fcb6fccd30cc2d'
    source_label = str(source_roots[0]) + ' (Kaggle-expanded archive)'
os.chdir(work_root)
print('GPU:', torch.cuda.get_device_name(0))
print('Source:', source_label)
print('Source SHA-256:', actual_sha)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle-main.txt'], check=True)

In [ ]:
input_root = Path('/kaggle/input')
swat_files = list(input_root.rglob('SWaT_Dataset_Normal_v1.xlsx'))
wadi_files = list(input_root.rglob('WADI_14days_new.csv'))
assert len(swat_files) == 1, swat_files
assert len(wadi_files) == 1, wadi_files
swat_root = swat_files[0].parents[2]
wadi_root = wadi_files[0].parents[2]
os.environ['MTAD_GAT_SWAT_ROOT'] = str(swat_root)
os.environ['MTAD_GAT_WADI_ROOT'] = str(wadi_root)
os.environ['MTAD_GAT_DATASETS_ROOT'] = str(work_root / 'datasets')
os.environ['MTAD_GAT_RUNS_ROOT'] = str(work_root / 'runs')
print('SWaT root:', swat_root)
print('WADI root:', wadi_root)

In [ ]:
subprocess.run([sys.executable, 'run.py', 'preprocess', '--dataset', 'SWAT'], check=True)
subprocess.run([sys.executable, 'run.py', 'preprocess', '--dataset', 'WADI'], check=True)
# 训练阶段只读取 /kaggle/working 中刚生成的 processed 产物。
os.environ['MTAD_GAT_SWAT_ROOT'] = str(work_root / 'datasets/SWAT')
os.environ['MTAD_GAT_WADI_ROOT'] = str(work_root / 'datasets/WADI')
assert (Path(os.environ['MTAD_GAT_SWAT_ROOT']) / 'processed/SWAT_train.pkl').is_file()
assert (Path(os.environ['MTAD_GAT_WADI_ROOT']) / 'processed/WADI_train.npy').is_file()

In [ ]:
plan = 'configs/internal/89_swat_wadi_frozen_c3_restricted_film.json'
subprocess.run([sys.executable, 'run.py', 'internal', '--plan', plan, '--dry-run'], check=True)
subprocess.run([sys.executable, '-m', 'src.runners.compare_experiments', '--plan', plan, '--batch-tag', 'kaggle_v1'], check=True)

In [ ]:
import shutil
run_root = Path('runs/internal/89_swat_wadi_frozen_c3_restricted_film__kaggle_v1')
assert (run_root / 'run_registry.json').is_file(), run_root
archive = shutil.make_archive('/kaggle/working/89_swat_wadi_frozen_c3_restricted_film__kaggle_v1', 'zip', run_root)
print('Results:', archive)